# 05_batch.ipynb — UC-5 일괄 수행

`batch()` 함수와 T-18 일괄 수행 흐름, 순서 유지, 예외 검증.


In [ ]:
import os
import json
import youtube_kit as yk

from pathlib import Path
# 루트의 outputs 에 저장한다. notebooks 에서 열어도, 루트에서 실행해도 같은 곳이다
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
out_dir = str(ROOT / "outputs")
os.makedirs(out_dir, exist_ok=True)
print("youtube_kit version:", yk.__version__)


## 1. T-18 흐름: 재생목록 조회


In [2]:
playlist_url = "https://www.youtube.com/playlist?list=PLyPc3E0Xm540Yfdln9qlVyyrqRME80jJm"
p_info = yk.info(playlist_url, out_dir=out_dir)
playlist_file = os.path.join(out_dir, f"{p_info['id']}.playlist.json")
assert os.path.exists(playlist_file)
print(f"Playlist: {p_info['title']} - {len(p_info['entries'])} entries")


Playlist: AI - 103 entries


## 2. 영상마다 info() 조회 및 job 작성
접근 불가 2편(Et9xVTOjgko, 9_kPesKDlE4)과 정상 영상들을 포함하여 job 목록 구성


In [3]:
# T-18의 대표 영상들과 접근 불가 2편을 포함한 영상 목록으로 job 구성
# 접근 불가 2편: Et9xVTOjgko (비공개), 9_kPesKDlE4 (멤버십)
# 정상 영상들: p_info['entries']의 대표 영상들
target_entries = [
    p_info["entries"][0],
    next(e for e in p_info["entries"] if e["id"] == "Et9xVTOjgko"),  # 접근 불가 1 (비공개)
    next(e for e in p_info["entries"] if e["id"] == "9_kPesKDlE4"),   # 접근 불가 2 (멤버십)
    p_info["entries"][1],
    p_info["entries"][2],
]

urls = []
jobs = []

for entry in target_entries:
    vid = entry["id"]
    std_url = entry["url"]
    urls.append(std_url)
    try:
        vid_info = yk.info(std_url, out_dir=out_dir)
        # info() 결과에서 오디오 포맷 선택 (하드코딩 금지)
        a_id = vid_info["audio_formats"][0]["id"]
        jobs.append({"do": "audio", "audio": a_id})
    except yk.VideoUnavailable as exc:
        print(f"Entry {vid} correctly detected as unavailable: {type(exc).__name__}")
        jobs.append({"do": "audio", "audio": "140"})

print(f"Prepared {len(urls)} urls and {len(jobs)} jobs.")


Entry Et9xVTOjgko correctly detected as unavailable: VideoUnavailable
Entry 9_kPesKDlE4 correctly detected as unavailable: VideoUnavailable
Prepared 5 urls and 5 jobs.


ERROR: [youtube] Et9xVTOjgko: Private video
ERROR: [youtube] 9_kPesKDlE4: This video is available to this channel's members on level: 슈퍼 코더 (or any higher level). Join this channel to get access to members-only content and other exclusive perks.


## 3. 일괄 수행 (batch)


In [4]:
batch_res = yk.batch(urls, jobs, out_dir=out_dir)
print("Batch summary:", f"ok={batch_res['ok']}, reused={batch_res['reused']}, failed={batch_res['failed']}")
for i, item in enumerate(batch_res["items"]):
    print(f"[{i+1}/{len(urls)}] {item['status']:<6} {item['video_id']} {item['do']}")

# 검증 1: items가 입력과 같은 순서
assert len(batch_res["items"]) == len(urls)
for item, expected_url in zip(batch_res["items"], urls):
    assert item["url"] == expected_url

# 검증 2: 모든 줄에 video_id와 url이 있음
for item in batch_res["items"]:
    assert "video_id" in item and item["video_id"] is not None
    assert "url" in item and item["url"] is not None

# 검증 3: 접근 불가 2편은 failed, 나머지는 ok 또는 reused
failed_items = [it for it in batch_res["items"] if it["status"] == "failed"]
assert len(failed_items) == 2
failed_ids = {it["video_id"] for it in failed_items}
assert failed_ids == {"Et9xVTOjgko", "9_kPesKDlE4"}
for it in failed_items:
    assert it["error"] == "VideoUnavailable"

success_items = [it for it in batch_res["items"] if it["status"] in ("ok", "reused")]
assert len(success_items) == len(urls) - 2

# 검증 4: 요약 건수가 items와 일치
assert batch_res["ok"] + batch_res["reused"] + batch_res["failed"] == len(batch_res["items"])
assert batch_res["failed"] == 2
print("T-18 batch verification passed successfully!")


Batch summary: ok=0, reused=3, failed=2
[1/5] reused lF8_DX2NxjI audio
[2/5] failed Et9xVTOjgko audio
[3/5] failed 9_kPesKDlE4 audio
[4/5] reused yL-CXA-5B4o audio
[5/5] reused VuCodghljNU audio
T-18 batch verification passed successfully!


ERROR: [youtube] Et9xVTOjgko: Private video
ERROR: [youtube] 9_kPesKDlE4: This video is available to this channel's members on level: 슈퍼 코더 (or any higher level). Join this channel to get access to members-only content and other exclusive perks.


## 4. 예외 검증 (InvalidInput)


In [5]:
# 길이 불일치
try:
    yk.batch(["bWPXADZylm0"], [], out_dir=out_dir)
    assert False, "Expected InvalidInput for length mismatch"
except yk.InvalidInput as exc:
    print("Length mismatch verified:", type(exc).__name__)

# info job 포함
try:
    yk.batch(["bWPXADZylm0"], [{"do": "info"}], out_dir=out_dir)
    assert False, "Expected InvalidInput for info job"
except yk.InvalidInput as exc:
    print("Info job rejection verified:", type(exc).__name__)

# 잘못된 do 액션
try:
    yk.batch(["bWPXADZylm0"], [{"do": "unknown"}], out_dir=out_dir)
    assert False, "Expected InvalidInput for unknown do"
except yk.InvalidInput as exc:
    print("Unknown action rejection verified:", type(exc).__name__)


Length mismatch verified: InvalidInput
Info job rejection verified: InvalidInput
Unknown action rejection verified: InvalidInput
